In [ ]:
!git clone https://github.com/varaiitj2527/PRMLProject.git
%cd PRMLProject

Cloning into 'PRMLProject'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 217 (delta 84), reused 36 (delta 9), pack-reused 18 (from 2)
Receiving objects: 100% (217/217), 211.79 MiB | 12.66 MiB/s, done.
Resolving deltas: 100% (84/84), done.
Updating files: 100% (53/53), done.
Filtering content: 100% (16/16), 1.40 GiB | 52.15 MiB/s, done.
/content/PRMLProject


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle

In [ ]:
def unpickle(file):
    with open(file, 'rb') as f:
        return pickle.load(f)

In [ ]:
train_RawData = unpickle('PreProcessedData/RawPixels_train.pkl')
test_RawData = unpickle('PreProcessedData/RawPixels_test.pkl')

train_labels = unpickle('PreProcessedData/Labels_train.pkl')
test_labels = unpickle('PreProcessedData/Labels_test.pkl')

train_PcaData = unpickle('PreProcessedData/TrainFeaturesPCA.pkl')
test_PcaData = unpickle('PreProcessedData/TestFeaturesPCA.pkl')

train_HoGData = unpickle('PreProcessedData/TrainFeaturesHoG.pkl')
test_HoGData = unpickle('PreProcessedData/TestFeaturesHoG.pkl')

train_Pca_HoGData = unpickle('PreProcessedData/TrainFeaturesPCA_HoG.pkl')
test_Pca_HoGData = unpickle('PreProcessedData/TestFeaturesPCA_HoG.pkl')

train_HoG_PcaData = unpickle('PreProcessedData/TrainFeaturesHoG_PCA.pkl')
test_HoG_PcaData = unpickle('PreProcessedData/TestFeaturesHoG_PCA.pkl')

resnet_train_data = unpickle('PreProcessedData/Resnet_train.pkl')
resnet_test_data = unpickle('PreProcessedData/Resnet_test.pkl')

train_ResNetData = np.array(resnet_train_data)
test_ResNetData = np.array(resnet_test_data)

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def train_xgboost(xtrain, ytrain, xval=None, yval=None, num_classes=10, params=None, num_rounds=300, early_stopping_rounds=5):
    """
    Trains a multi-class XGBoost model using GPU.
    """
    if params is None:
        params = {
          'objective': 'multi:softprob',
          'num_class': 10,
          'eval_metric': 'merror',
          'tree_method': 'hist',
          'device': 'cuda',
          'max_depth': 6,
          'eta': 0.05,
          'subsample': 0.9,
          'colsample_bytree': 0.9,
          'lambda': 2.0,
          'alpha': 0.5,
          'verbosity': 1
        }


    dtrain = xgb.DMatrix(xtrain, label=ytrain)

    if xval is not None and yval is not None:
        dval = xgb.DMatrix(xval, label=yval)
        evals = [(dtrain, 'train'), (dval, 'eval')]
        model = xgb.train(params, dtrain, num_boost_round=num_rounds,
                          evals=evals, early_stopping_rounds=early_stopping_rounds)
    else:
        model = xgb.train(params, dtrain, num_boost_round=num_rounds)

    return model

def predict_xgboost(model, xtest):

    dtest = xgb.DMatrix(xtest)
    yprob = model.predict(dtest)
    ypred = yprob.argmax(axis=1)
    return ypred

def test_accuracy(ypred, ytest):

    return accuracy_score(ytest, ypred)


In [ ]:
model = train_xgboost_multiclass(train_HoG_PcaData, train_labels)

ypred = predict_xgboost_multiclass(model, test_HoG_PcaData)

acc = test_accuracy(ypred, test_labels)
print(f"Test Accuracy: {acc:.4f}")

Test Accuracy: 0.5746


In [ ]:
model_ = train_xgboost(train_ResNetData, train_labels)

ypred_ = predict_xgboost(model_, test_ResNetData)

acc_ = test_accuracy(ypred_, test_labels)
print(f"Test Accuracy: {acc_:.4f}")

Test Accuracy: 0.8770


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

def tune_xgboost(xtrain, ytrain, num_classes=10):
    """
    Tune XGBoost hyperparameters using GridSearchCV for multiclass classification.
    """
    model = XGBClassifier(
        objective='multi:softprob',
        num_class=num_classes,
        tree_method='hist',
        device='cuda',
        use_label_encoder=False,
        eval_metric='mlogloss'
    )

    param_grid = {
        'max_depth': [4, 6],
        'learning_rate': [0.1, 0.3],
        'subsample': [0.8, 1],
        'colsample_bytree': [0.8, 1],
        'n_estimators': [100, 200],
    }

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='accuracy',
        cv=3,
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(xtrain, ytrain)

    print("Best Parameters:", grid_search.best_params_)
    print("Best Accuracy on CV:", grid_search.best_score_)

    return grid_search.best_estimator_


In [ ]:
best_model = tune_xgboost(train_HoG_PcaData, train_labels, num_classes=10)

ypredt = best_model.predict(test_HoG_PcaData)
acct = accuracy_score(test_labels, ypredt)

print(f"Test Accuracy with Tuned Model: {acc:.4f}")

Fitting 3 folds for each of 32 candidates, totalling 96 fits


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:28:23] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.3, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1}
Best Accuracy on CV: 0.5702800608574412
Test Accuracy with Tuned Model: 0.5667


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:28:36] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [ ]:
print(f"Test Accuracy with Tuned Model: {acct:.4f}")

Test Accuracy with Tuned Model: 0.5802


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 19.8 MB/s eta 0:00:00


In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
import numpy as np

def objective(trial, xtrain, ytrain, num_classes):
    """
    Objective function for Optuna to optimize XGBoost hyperparameters.
    """
    params = {
        'objective': 'multi:softprob',
        'num_class': num_classes,
        'tree_method': 'hist',
        'device': 'cuda',
        'n_jobs': -1,
        'verbosity': 0,
        'eval_metric': 'mlogloss',
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
    }

    model = XGBClassifier(**params)
    score = cross_val_score(model, xtrain, ytrain, cv=3, scoring='accuracy', n_jobs=-1)
    return score.mean()

def tune_xgboost_optuna(xtrain, ytrain, num_classes=10, n_trials=30):
    """
    Runs Optuna hyperparameter tuning for multiclass XGBoost.
    """
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, xtrain, ytrain, num_classes), n_trials=n_trials, n_jobs=-1)

    print("Best Trial:")
    print("  Value: ", study.best_value)
    print("  Params: ", study.best_params)

    best_model = XGBClassifier(
        objective='multi:softprob',
        num_class=num_classes,
        tree_method='hist',
        device='cuda',
        n_jobs=-1,
        eval_metric='mlogloss',
        **study.best_params
    )
    best_model.fit(xtrain, ytrain)
    return best_model


In [ ]:
modelo = tune_xgboost_optuna(train_HoG_PcaData, train_labels, num_classes=10, n_trials=30)

ypredo = modelo.predict(test_HoG_PcaData)
acco = accuracy_score(test_labels, ypredo)
print(f"Test Accuracy with Optuna-Tuned XGBoost: {acco:.4f}")

[I 2025-04-08 19:29:20,675] A new study created in memory with name: no-name-0e7fb83b-5b93-420b-af3a-b0bb89c26de8
[I 2025-04-08 19:30:16,906] Trial 0 finished with value: 0.5605000116486331 and parameters: {'max_depth': 7, 'learning_rate': 0.08136912974576203, 'subsample': 0.7677381522791562, 'colsample_bytree': 0.6697421402851351, 'gamma': 1.2392426277398882, 'min_child_weight': 4, 'n_estimators': 146}. Best is trial 0 with value: 0.5605000116486331.
[I 2025-04-08 19:30:35,759] Trial 1 finished with value: 0.5535200096430088 and parameters: {'max_depth': 7, 'learning_rate': 0.2691412794787656, 'subsample': 0.733478358589329, 'colsample_bytree': 0.7124889913431895, 'gamma': 3.305195048947884, 'min_child_weight': 10, 'n_estimators': 207}. Best is trial 0 with value: 0.5605000116486331.
[I 2025-04-08 19:31:26,537] Trial 2 finished with value: 0.5649000268524571 and parameters: {'max_depth': 6, 'learning_rate': 0.08349503214604523, 'subsample': 0.7965364966214723, 'colsample_bytree': 0.75

Best Trial:
  Value:  0.5768400328621293
  Params:  {'max_depth': 6, 'learning_rate': 0.17159425933411315, 'subsample': 0.8933102924581244, 'colsample_bytree': 0.9154897675403644, 'gamma': 0.026369966624579647, 'min_child_weight': 7, 'n_estimators': 272}
Test Accuracy with Optuna-Tuned XGBoost: 0.5844


In [ ]:
m_, s_ = tune_xgboost_optuna(train_ResNetData, train_labels, 10, 20)
predict(m_, test_ResNetData, test_labels)

[I 2025-04-10 07:10:44,452] A new study created in memory with name: no-name-2ec0d130-780b-4b36-a788-f6521c79f214
[I 2025-04-10 07:18:59,913] Trial 0 finished with value: 0.8735 and parameters: {'max_depth': 10, 'learning_rate': 0.29382111802150146, 'subsample': 0.6920517124948876, 'colsample_bytree': 0.7053637197971317, 'gamma': 2.017199550901907, 'min_child_weight': 10}. Best is trial 0 with value: 0.8735.
[I 2025-04-10 07:20:00,804] Trial 1 finished with value: 0.8708 and parameters: {'max_depth': 9, 'learning_rate': 0.21390351316627595, 'subsample': 0.8780407999794593, 'colsample_bytree': 0.9821869561387805, 'gamma': 1.6675310413753115, 'min_child_weight': 6}. Best is trial 1 with value: 0.8708.
[I 2025-04-10 07:20:30,014] Trial 1 finished with value: 0.8812 and parameters: {'max_depth': 6, 'learning_rate': 0.07371010243859646, 'subsample': 0.6387309899148957, 'colsample_bytree': 0.7806746632525146, 'gamma': 4.159152808937244, 'min_child_weight': 8}. Best is trial 1 with value: 0.8

Best Accuracy: 0.8932
Best Parameters: {'max_depth': 5, 'learning_rate': 0.08350311741107516, 'subsample': 0.606096091970009, 'colsample_bytree': 0.7097050131139339, 'gamma': 0.8676513754278131, 'min_child_weight': 8}
Test Accuracy: 0.8639
